# Upload GGUF files to HuggingFace

Uploads:
- `gpt-oss-20b.MXFP4.gguf` — native MXFP4, ~13.8 GB
- `gpt-oss-20b-sft-IQ4_XS-hybrid.gguf` — hybrid IQ4_NL+IQ4_XS, ~11.6 GB

In [9]:
import os
from huggingface_hub import HfApi, login

# ── Configure these ──────────────────────────────────────────────────────────
HF_REPO_ID  = "ospost/gpt-oss-20b-sft-GGUF"  # change to your HF username
HF_TOKEN    = os.environ.get("HF_TOKEN", "")         # set env var or paste token here
PRIVATE     = False                                   # set True to keep repo private
# ─────────────────────────────────────────────────────────────────────────────



FILES = {
#    "gpt-oss-20b.MXFP4.gguf":           "MXFP4 native format (~13.8 GB)",
    "gpt-oss-20b-sft-IQ4_XS-hybrid.gguf": "Hybrid: MXFP4 experts + IQ4_XS attention (~11.6 GB)",
}

def file_size(path):
    if not os.path.exists(path): return "❌ not found"
    return f"{os.path.getsize(path)/1e9:.1f} GB"

print("Files to upload:")
for f, desc in FILES.items():
    print(f"  {f:<45} {file_size(f)}  — {desc}")

if not HF_TOKEN:
    raise ValueError("Set HF_TOKEN environment variable or paste your token into HF_TOKEN above.")

Files to upload:
  gpt-oss-20b-sft-IQ4_XS-hybrid.gguf            12.1 GB  — Hybrid: MXFP4 experts + IQ4_XS attention (~11.6 GB)


In [10]:
# Login and create repo if it doesn't exist

login(token=HF_TOKEN)
api = HfApi(token=HF_TOKEN)

try:
    api.repo_info(repo_id=HF_REPO_ID, repo_type="model")
    print(f"✅ Repo already exists: https://huggingface.co/{HF_REPO_ID}")
except Exception:
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model", private=PRIVATE)
    print(f"✅ Created repo: https://huggingface.co/{HF_REPO_ID}")

✅ Repo already exists: https://huggingface.co/ospost/gpt-oss-20b-sft-GGUF


In [11]:
# Create a minimal model card describing the quantization

MODEL_CARD = f"""---
base_model: unsloth/gpt-oss-20b
tags:
  - gguf
  - quantized
  - mxfp4
  - iq4_xs
---

# GPT-OSS-20B SFT — GGUF

Fine-tuned GPT-OSS-20B (SFT) exported as GGUF for llama.cpp inference.

## Files

| File | Size | Description |
|---|---|---|
| `gpt-oss-20b.MXFP4.gguf` | ~13.8 GB | Native MXFP4 format — highest quality |
| `gpt-oss-20b-sft-IQ4_XS-hybrid.gguf` | ~11.6 GB | Hybrid: MXFP4 experts + IQ4_XS attention |

## Quantization Strategy

GPT-OSS-20B uses MXFP4 MoE expert weights that **cannot be requantized without quality loss**.
The hybrid GGUF keeps expert weights in their native MXFP4 format:

- `ffn_*_exps` (72 tensors) → `IQ4_NL` — MXFP4 preserved
- `attn_q/k` and other 2880-column tensors → `IQ4_NL` — avoids 256-block fallback
- All other tensors → `IQ4_XS` + imatrix calibration

## Usage

```bash
llama-cli --model gpt-oss-20b-sft-IQ4_XS-hybrid.gguf --n-gpu-layers 99 -p "Your prompt here"
```
"""

with open("README.md", "w") as f:
    f.write(MODEL_CARD)

api.upload_file(
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id=HF_REPO_ID,
    repo_type="model",
)
print("✅ Model card uploaded")

No files have been modified since last commit. Skipping to prevent empty commit.


✅ Model card uploaded


In [12]:
# Upload GGUF files
# huggingface_hub handles chunked upload and resume automatically for large files.

for local_path, description in FILES.items():
    if not os.path.exists(local_path):
        print(f"⚠️  Skipping {local_path} — not found")
        continue

    size_gb = os.path.getsize(local_path) / 1e9
    print(f"\nUploading {local_path} ({size_gb:.1f} GB) ...")

    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=local_path,
        repo_id=HF_REPO_ID,
        repo_type="model",
    )
    print(f"✅ {local_path} uploaded")

print(f"\n🎉 All done: https://huggingface.co/{HF_REPO_ID}")


Uploading gpt-oss-20b-sft-IQ4_XS-hybrid.gguf (12.1 GB) ...


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  gpt-oss-20b-sft-IQ4_XS-hybrid.gguf    :   0%|          | 12.9MB / 12.1GB            

✅ gpt-oss-20b-sft-IQ4_XS-hybrid.gguf uploaded

🎉 All done: https://huggingface.co/ospost/gpt-oss-20b-sft-GGUF


In [13]:
# Optional: verify uploaded files

from huggingface_hub import list_repo_files

print(f"Files in {HF_REPO_ID}:")
for f in list_repo_files(HF_REPO_ID, repo_type="model", token=HF_TOKEN):
    print(f"  {f}")

Files in ospost/gpt-oss-20b-sft-GGUF:
  .gitattributes
  README.md
  gpt-oss-20b-sft-IQ4_XS-hybrid.gguf
  gpt-oss-20b.MXFP4.gguf


In [14]:
# Upload quantization artifacts to the same GGUF model repo
# (imatrix, logs, calibration — useful for reproducibility)

ARTIFACTS = [
    "gpt-oss-20b-imatrix.dat",   # others can reuse for further quantization
    "calibration.txt",            # shows what data was used for imatrix
    "quantize_dry_run.log",       # tensor type discovery log
    "quantize_final.log",         # full quantization log
]

print("Uploading quantization artifacts to model repo ...")
for path in ARTIFACTS:
    if not os.path.exists(path):
        print(f"  ⚠️  Skipping {path} — not found")
        continue
    size_mb = os.path.getsize(path) / 1e6
    print(f"  Uploading {path} ({size_mb:.1f} MB) ...")
    api.upload_file(
        path_or_fileobj=path,
        path_in_repo=f"quantization/{path}",
        repo_id=HF_REPO_ID,
        repo_type="model",
    )
    print(f"  ✅ {path}")

print(f"\n✅ Artifacts uploaded to {HF_REPO_ID}/quantization/")

# ── Upload SFT dataset to a separate dataset repo ────────────────────────────
SFT_DATASET_FILE = "train_sft_final.json"
HF_DATASET_REPO  = HF_REPO_ID.replace("-GGUF", "-sft-dataset")  # e.g. user/gpt-oss-20b-sft-dataset

if os.path.exists(SFT_DATASET_FILE):
    print(f"\nUploading {SFT_DATASET_FILE} to dataset repo {HF_DATASET_REPO} ...")

    # Create dataset repo if needed
    try:
        api.repo_info(repo_id=HF_DATASET_REPO, repo_type="dataset")
        print("  Repo already exists")
    except Exception:
        api.create_repo(repo_id=HF_DATASET_REPO, repo_type="dataset", private=PRIVATE)
        print(f"  Created dataset repo: {HF_DATASET_REPO}")

    size_mb = os.path.getsize(SFT_DATASET_FILE) / 1e6
    print(f"  Uploading {SFT_DATASET_FILE} ({size_mb:.1f} MB) ...")
    api.upload_file(
        path_or_fileobj=SFT_DATASET_FILE,
        path_in_repo=SFT_DATASET_FILE,
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
    )
    print(f"  ✅ Done: https://huggingface.co/datasets/{HF_DATASET_REPO}")
else:
    print(f"\n⚠️  {SFT_DATASET_FILE} not found — skipping dataset upload")

Uploading quantization artifacts to model repo ...
  Uploading gpt-oss-20b-imatrix.dat (28.1 MB) ...


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  gpt-oss-20b-imatrix.dat               :   3%|2         |  794kB / 28.1MB            

  ✅ gpt-oss-20b-imatrix.dat
  Uploading calibration.txt (0.2 MB) ...
  ✅ calibration.txt
  Uploading quantize_dry_run.log (0.2 MB) ...
  ✅ quantize_dry_run.log
  Uploading quantize_final.log (0.2 MB) ...
  ✅ quantize_final.log

✅ Artifacts uploaded to ospost/gpt-oss-20b-sft-GGUF/quantization/

Uploading train_sft_final.json to dataset repo ospost/gpt-oss-20b-sft-sft-dataset ...
  Created dataset repo: ospost/gpt-oss-20b-sft-sft-dataset
  Uploading train_sft_final.json (109.9 MB) ...


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  train_sft_final.json                  :   2%|1         | 2.09MB /  110MB            

  ✅ Done: https://huggingface.co/datasets/ospost/gpt-oss-20b-sft-sft-dataset
